# QCoDeS in 15 mins: QTHz Lab version

## Imports

In [29]:
import numpy as np
from scipy.io import savemat
import yaml, logging, time, json, datetime as dt
from pathlib import Path

import qcodes as qc
from qcodes.station import Station
from qcodes.parameters import Parameter
from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_or_create_experiment
)
from qcodes.instrument import Instrument
from qcodes.instrument_drivers.stanford_research import SR860

## Setup logging

In [12]:
log = logging.getLogger("tutorial")
log.setLevel(logging.INFO)
log.propagate = False
for h in list(log.handlers):          # clear stale handlers on re-run
    log.removeHandler(h)
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-7s %(message)s", "%H:%M:%S"))
log.addHandler(h)

qc.logger.start_all_logging()
log.info("QCoDeS log files in: %s", qc.logger.get_log_file_name())

Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\qthzl\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\qthzl\.qcodes\logs\260911-12116-qcodes.log
12:42:12  INFO    QCoDeS log files in: C:\Users\qthzl\.qcodes\logs\260911-12116-qcodes.log


## Read Configuration

In [25]:
CONFIG_PATH = Path("config.yaml").resolve()
ROOT = CONFIG_PATH.parent
config_text = CONFIG_PATH.read_text()
cfg = yaml.safe_load(config_text)

log.info("loaded config: %s", 'path to config')


13:32:36  INFO    loaded config: path to config


## Database and Experiment

In [14]:
db_path = (ROOT / cfg["database"]).resolve()
db_path.parent.mkdir(parents=True, exist_ok=True)

initialise_or_create_database_at(db_path)
log.info("Database: %s (%.1f MB)", db_path, db_path.stat().st_size / 1e6)

exp = load_or_create_experiment(
    experiment_name = cfg["experiment_name"],
    sample_name = cfg['sample_name']
)
log.info("Experiment #%d '%s' on sample '%s' - %d run(s) so far", exp.exp_id, exp.name, exp.sample_name, len(exp.data_sets()))

12:42:35  INFO    Database: C:\Data\20260911\tutorial.db (0.0 MB)
12:42:35  INFO    Experiment #1 'QCoDeS_Tutorial' on sample 'demo_sample' - 0 run(s) so far


## Instruments and Station

In [26]:
Instrument.close_all()
station = Station()

lck_cfg = cfg['instruments']['lockin']

t0 = time.perf_counter()
lockin = SR860("lockin", lck_cfg['address'])
log.info("Connected to %s at %s in %.2f s",
         lockin.IDN(), lck_cfg['address'], time.perf_counter()-t0)

for pname, requested in lck_cfg.get("settings", {}).items():
    param = lockin.parameters[pname]
    param(requested)
    time.sleep(0.5)
    actual = param()
    flag = "" if actual == requested else "  <-- DIFFERS from requested %r" % requested
    log.info("%-16s = %-10r %s%s", pname, actual, param.unit, flag)

station.add_component(lockin)
log.info("Station components: %s", list(station.components))

Connected to: Stanford_Research_Systems SR860 (serial:007339, firmware:1.55) in 0.03s
13:33:35  INFO    Connected to {'vendor': 'Stanford_Research_Systems', 'model': 'SR860', 'serial': '007339', 'firmware': '1.55'} at GPIB0::4::INSTR in 0.04 s
13:33:35  INFO    frequency        = 17.777     Hz
13:33:36  INFO    amplitude        = 0.10000000149 V  <-- DIFFERS from requested 0.1
13:33:36  INFO    time_constant    = 0.1        s
13:33:37  INFO    sensitivity      = 0.01       V
13:33:37  INFO    input_config     = 'a'        
13:33:38  INFO    input_coupling   = 'ac'       
13:33:38  INFO    input_shield     = 'ground'   
13:33:39  INFO    Station components: ['lockin']


## Register parameters

In [21]:
elapsed_time = Parameter(
    "elapsed_time",
    label = "Time",
    unit = "s",
    set_cmd = None,
    get_cmd = None
)

meas = Measurement(
    exp = exp,
    station = station,
    name = "QCoDeS_tutorial"
)
meas.write_period = 1.0

meas.register_parameter(elapsed_time)
recorded = [lockin.X, lockin.Y, lockin.R, lockin.P]
for p in recorded:
    meas.register_parameter(p, setpoints = (elapsed_time,))

log.info("Registered %d columns:", len(meas.parameters))

13:14:23  INFO    Registered 5 columns:


## Measurement

In [27]:
mcfg = cfg["measurement"]
duration = mcfg["duration"]
interval = mcfg["sample_interval"]
tau = lockin.time_constant()

time.sleep(3*tau)

with meas.run() as datasaver:
    ds = datasaver.dataset
    log.info("Run #%d started (guid %s)", ds.run_id, ds.guid)

    n = 0
    t_start = time.perf_counter()
    while True:
        t = time.perf_counter() - t_start
        if t > duration:
            break
        x, y = lockin.get_values("X", "Y")
        r, p = lockin.get_values("R", "P")

        datasaver. add_result(
            (elapsed_time, t),
            (lockin.X, x),
            (lockin.Y, y),
            (lockin.R, r),
            (lockin.P, p)
        )

        n += 1
        if n % max(1, int(1.0 / interval)) == 0:
            log.info("t = %6.2f s   X = %+.4e   Y=%+.4e", t, x, y)

        time.sleep(max(0.0, interval - ((time.perf_counter() - t_start) - t)))

log.info("Run #%d finished: %d points in %.2f s (%.1f Hz)", ds.run_id, n, t, n/t)

Starting experimental run with id: 3. 
13:33:55  INFO    Run #3 started (guid 3d494ee1-0000-0000-0000-01a0918877fe)
13:33:56  INFO    t =   0.90 s   X = +9.4324e-09   Y=-2.7692e-07
13:33:57  INFO    t =   1.91 s   X = +1.6239e-07   Y=+5.2150e-07
13:33:58  INFO    t =   2.91 s   X = -6.1044e-07   Y=-6.7537e-07
13:33:59  INFO    t =   3.92 s   X = +8.0220e-06   Y=-1.7377e-04
13:34:00  INFO    t =   4.92 s   X = +1.1276e-04   Y=+3.1879e-05
13:34:01  INFO    t =   5.93 s   X = +5.6423e-07   Y=+7.3342e-07
13:34:02  INFO    t =   6.93 s   X = +1.3909e-07   Y=-1.5158e-07
13:34:03  INFO    t =   7.93 s   X = -3.2705e-07   Y=+2.3453e-08
13:34:04  INFO    t =   8.94 s   X = -1.7212e-07   Y=-1.8147e-07
13:34:05  INFO    t =   9.94 s   X = -6.5273e-07   Y=-1.4056e-08
13:34:05  INFO    Run #3 finished: 100 points in 10.04 s (10.0 Hz)


## Matlab data format

In [31]:
export_dir = (ROOT / cfg["export_dir"]).resolve()
export_dir.mkdir(parents=True, exist_ok=True)

df = ds.to_pandas_dataframe().reset_index()
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
mat_path = export_dir / f"run{ds.run_id:04}_{ds.guid[:8]}_{stamp}.mat"

mat = {col: df[col].to_numpy() for col in df.columns}
mat.update({
    "run_id": ds.run_id,
    "guid": ds.guid,
    "experiment": exp.name,
    "sample": exp.sample_name,
    "config_yaml": config_text,
    "snapshot_json": json.dumps(ds.snapshot),
    "units": {p.name: p.unit for p in [elapsed_time, *recorded]}
})

savemat(mat_path, mat, do_compression=True)
log.info("MATLAB file: %s (%.1f kB, %d rows)",
         mat_path.name, mat_path.stat().st_size / 1e3, len(df))

13:49:11  INFO    MATLAB file: run0003_3d494ee1_20260911_134911.mat (5.5 kB, 100 rows)
